In [0]:
# Robot fleet health monitoring - synthetic data generation (fixed seed => reproducible)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SEED = 42
TARGET = "workspace.default"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET}")

# ------------------------------------------------------------------ 1. robots (500 rows)
models = ["RX-100", "RX-200", "RX-300"]
sites = ["Warehouse North", "Warehouse South", "Warehouse East", "Warehouse West", "Warehouse Central"]

robots = (
    spark.range(500)
    .select(
        F.col("id").cast("int").alias("robot_id"),
        F.element_at(F.array(*[F.lit(m) for m in models]), (F.rand(SEED) * 3).cast("int") + 1).alias("model"),
        F.element_at(F.array(*[F.lit(s) for s in sites]), (F.rand(SEED + 1) * 5).cast("int") + 1).alias("site"),
        F.date_add(F.lit("2025-01-01").cast("date"), (F.rand(SEED + 2) * 365).cast("int")).alias("deploy_date"),
    )
)
robots.write.mode("overwrite").saveAsTable(f"{TARGET}.robots")

# ------------------------------------------------------------- 2. failure_log (40 rows)
failure_types = ["overheat", "localization_drift", "battery_drop"]

failures = (
    robots.select("robot_id")
    # deterministic shuffle -> 40 distinct robots
    .withColumn("failure_id", F.row_number().over(Window.orderBy(F.rand(SEED + 10))))
    .filter(F.col("failure_id") <= 40)
    .withColumn(
        "failure_type",
        F.element_at(F.array(*[F.lit(t) for t in failure_types]), (F.rand(SEED + 11) * 3).cast("int") + 1),
    )
    # window length: 2-12 hours
    .withColumn("duration_s", ((2 + F.rand(SEED + 12) * 10) * 3600).cast("long"))
    # start chosen so the whole window lies inside the telemetry range (Oct 2026)
    .withColumn(
        "start_unix",
        F.unix_timestamp(F.lit("2026-10-01 00:00:00"))
        + (F.rand(SEED + 13)
           * (F.unix_timestamp(F.lit("2026-10-31 23:59:59")) - F.col("duration_s")
              - F.unix_timestamp(F.lit("2026-10-01 00:00:00")))).cast("long"),
    )
    .select(
        F.col("failure_id").cast("int").alias("failure_id"),
        F.col("robot_id").cast("int").alias("robot_id"),
        F.timestamp_seconds(F.col("start_unix")).alias("start_ts"),
        F.timestamp_seconds(F.col("start_unix") + F.col("duration_s")).alias("end_ts"),
        F.col("failure_type"),
    )
)
failures.write.mode("overwrite").saveAsTable(f"{TARGET}.failure_log")

print(f"robots and failure_log written to {TARGET}.")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


robots and failure_log written to workspace.default.


In [0]:
# --------------------------------------- 3. bronze_telemetry (every 5 min, Oct 2026)
TELEMETRY_START = "2026-10-01 00:00:00"
TELEMETRY_END = "2026-10-31 23:59:59"
N_INTERVALS = 31 * 24 * 12  # 5-minute grid: 8,928 slots/robot

time_grid = (
    spark.range(N_INTERVALS)
    .select(F.timestamp_seconds(F.unix_timestamp(F.lit(TELEMETRY_START)) + F.col("id") * 300).alias("ts"))
)

robots_df = spark.table(f"{TARGET}.robots")
failures_df = spark.table(f"{TARGET}.failure_log")

# one row per robot per 5-minute slot (crossJoin); attach model + failure window
telemetry = (
    robots_df.select("robot_id", "model")
    .crossJoin(time_grid)
    .join(failures_df, "robot_id", "left")
)

frac_hour = F.hour("ts") + F.minute("ts") / 60.0
sine = F.sin(2 * 3.141592653589793 * frac_hour / 24)  # daily cycle

in_window = (
    F.col("start_ts").isNotNull()
    & (F.col("ts") >= F.col("start_ts"))
    & (F.col("ts") < F.col("end_ts"))
)
progress = F.when(
    in_window,
    (F.unix_timestamp("ts") - F.unix_timestamp("start_ts"))
    / (F.unix_timestamp("end_ts") - F.unix_timestamp("start_ts")),
).otherwise(F.lit(0.0))

# base signal + daily sine cycle + randn noise
battery_base = F.least(F.lit(100.0), F.greatest(F.lit(5.0), 92 + 5 * sine + F.randn(SEED + 20) * 1.5))
motor_base = 58 + 8 * sine + F.randn(SEED + 21) * 2.5
torque_base = 42 + 6 * sine + F.randn(SEED + 22) * 2.0
loc_base = F.greatest(F.lit(0.1), 1.8 + 0.6 * sine + F.randn(SEED + 23) * 0.3)
speed_base = F.greatest(F.lit(0.0), 1.5 + 0.4 * sine + F.randn(SEED + 24) * 0.15)

# failure effects during the robot's window
#   battery_drop: falls sharply in the first third, then stays low
battery = F.when(
    (F.col("failure_type") == "battery_drop") & in_window,
    F.greatest(F.lit(3.0), battery_base * (1 - 0.75 * F.least(progress * 3, F.lit(1.0)))),
).otherwise(battery_base)
#   overheat: motor temp rises gradually, up to +35 C by window end
motor = F.when(
    (F.col("failure_type") == "overheat") & in_window,
    motor_base + 35 * progress,
).otherwise(motor_base)
#   localization_drift: loc error grows steadily, up to 10x normal
loc = F.when(
    (F.col("failure_type") == "localization_drift") & in_window,
    loc_base * (1 + 9 * progress),
).otherwise(loc_base)

# data-quality: RX-300 reports motor temp in Fahrenheit (kept in motor_temp_c on purpose)
motor = F.when(F.col("model") == "RX-300", motor * 9 / 5 + 32).otherwise(motor)


def null_2pct(col, seed):
    """Data-quality: set ~2% of a sensor's values to null."""
    return F.when(F.rand(seed) < 0.02, F.lit(None).cast("double")).otherwise(col)


telemetry = (
    telemetry.select(
        "robot_id",
        "ts",
        null_2pct(battery, SEED + 30).alias("battery_pct"),
        null_2pct(motor, SEED + 31).alias("motor_temp_c"),
        null_2pct(torque_base, SEED + 32).alias("joint_torque_nm"),
        null_2pct(loc, SEED + 33).alias("loc_error_cm"),
        null_2pct(speed_base, SEED + 34).alias("speed_mps"),
    )
    # data-quality: ~1% of rows duplicated
    .withColumn("_n", F.when(F.rand(SEED + 40) < 0.01, F.lit(2)).otherwise(F.lit(1)))
    .withColumn("_dup", F.explode(F.sequence(F.lit(1), F.col("_n"))))
    .drop("_n", "_dup")
)

telemetry.write.mode("overwrite").saveAsTable(f"{TARGET}.bronze_telemetry")

print(f"bronze_telemetry written to {TARGET}.")

bronze_telemetry written to workspace.default.


In [0]:
# ------------------------------------------------------------------ 4. verify
for t in ["robots", "failure_log", "bronze_telemetry"]:
    print(f"{TARGET}.{t}: {spark.table(f'{TARGET}.{t}').count()} rows")

display(spark.table(f"{TARGET}.bronze_telemetry").orderBy("robot_id", "ts").limit(10))

workspace.default.robots: 500 rows
workspace.default.failure_log: 40 rows
workspace.default.bronze_telemetry: 4508231 rows


robot_id,ts,battery_pct,motor_temp_c,joint_torque_nm,loc_error_cm,speed_mps
0,2026-10-01T00:00:00.000Z,89.59899709537194,60.461033822299804,43.56520404648112,1.7892755450034705,1.560409544220792
0,2026-10-01T00:05:00.000Z,null,59.053592612388755,39.0088756315193,1.1174009801961056,1.3318007493847186
0,2026-10-01T00:10:00.000Z,92.56846598375233,58.65387474289627,47.03217638788623,2.350636795504177,1.4044721797007782
0,2026-10-01T00:15:00.000Z,93.02193964458783,53.19135337108856,43.60102851178228,2.0922538282948535,1.2871460555513436
0,2026-10-01T00:20:00.000Z,93.84910625757774,59.00451859928186,44.58418279184376,1.9009872946544306,1.5319094339119086
0,2026-10-01T00:25:00.000Z,92.33261423279242,57.8439444174895,42.41729065130733,2.12385271862279,1.8010261835133914
0,2026-10-01T00:30:00.000Z,92.35148224209222,null,44.524761677679756,1.7808843178988425,1.8883158763060215
0,2026-10-01T00:35:00.000Z,92.14585126319321,61.37199544787764,44.409794053522496,1.9101259297458346,1.5312163961660257
0,2026-10-01T00:40:00.000Z,91.98554610869071,59.270647742622295,41.77088478526615,2.2618973177227555,1.5943859462408585
0,2026-10-01T00:45:00.000Z,92.58789595034678,59.80933939422962,42.16267983388054,2.275330269424399,1.6338004439881229
